## PKLM + RB

- 2026-06-16
- jw

In [1]:
library(PKLMtest)
library(tidyverse)
library(caret)
library(RBtest)

Loading required package: parallel
Loading required package: ranger
ranger 0.18.0 using 2 threads (default). Change with num.threads in ranger() and predict(), options(Ncpus = N), options(ranger.num.threads = N) or environment variable R_RANGER_NUM_THREADS.
── Attaching core tidyverse packages ─────────────────────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     
── Conflicts ───────────────────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the ]8;;http://conflicted.r-lib.org/conflicted package]8;; to force all conflicts to become errors
Loading required package: lattice

Attaching package: ‘caret’

The following object is masked from ‘package:purrr’:

    lift



In [ ]:
#source('miss_helpers.R')

In [ ]:
# ── 1. 載入資料 ─────────────────────────────────────────────
dat <- read_csv("miss36.csv")
#dat <- read_csv("miss72.csv")

#' 這一步很重要，要針對不同題本剔除掉不同的跳答題。
#' 算了，直接從 >50% missing 刪除就好，以免刪掉太多。
#skip_vars <- skip_items$m36

#' NA 值重編碼
missing_codes <- c(7777, 8888, 9996, 9999)
dat <- dat %>%
  mutate(across(everything(), ~ifelse(. %in% missing_codes, NA, .)))

Rows: 2164 Columns: 700
── Column specification ───────────────────────────────────────────────────────────────────────
Delimiter: ","
chr  (30): release_id, pfa03a, pfa04a, pfa060108a, pfa1501a, pfa1502a, pfb0120a, pfb0502a, ...
dbl (665): interviewer_id, baby_sex, baby_doby, baby_dobm, int_y, int_m, int_months, relati...
lgl   (5): pfb0802a, famedu1401a, famedu1402a, famedu1403a, attrition

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


## Step 0 Data Cleaning

In [ ]:
library(dplyr)

drop_items <- c(
  # 完訪時間
  'int_months', 'int_y', 'int_m',
  # 幼兒出生年月
  'baby_doby', 'baby_dobm',
  # 身長/體重測量時間（當期）
  'heighy', 'heighm', 'heighd',
  'weighty', 'weightm', 'weightd',
  # 出生體重/身長
  'postn03',
  # 行政識別與加權
  'wsel0', 'w_post_sel', 'w_raking_sel',
  'release_id', 'interviewer_id',
  # 訪問形式
  'video',
  # 父母/照顧者出生年
  'pfa0101', 'pfa0101a',
  'pfa0102', 'pfa0102a',
  'pfc01'
)

dat_clean <- dat |>
  select(-any_of(drop_items)) |>
  select(-matches("^(growheigh|groweight|growy|growm|growd)\\d+$"))

In [ ]:
# ── 3. 移除 target, 跳答題 ─────────────────────────
X <- dat %>%
  select(-attrition) %>%
  mutate(across(where(is.factor),    as.integer)) %>%
  mutate(across(where(is.character), as.integer)) %>%
  # 只需要這一步：刪子題，守門題自動保留
  select(-any_of(drop_items)) %>%
  #select(-any_of(skip_vars)) %>%
  select(where(is.numeric))                 # 必須是 matrix

cat("矩陣維度：", nrow(X), "×", ncol(X), "\n")


Warning message:
There were 27 warnings in `mutate()`.
The first warning was:
ℹ In argument: `across(where(is.character), as.integer)`.
Caused by warning:
! NAs introduced by coercion
ℹ Run ]8;;x-r-run:dplyr::last_dplyr_warnings()dplyr::last_dplyr_warnings()]8;; to see the 26 remaining warnings. 


矩陣維度： 2164 × 399 


In [9]:
# ── 3. 缺值率篩選（門檻 0.5）────────────────────────────────
missing_rate <- colMeans(is.na(X))
X <- X[, missing_rate < 0.5]
cat(sprintf("缺值率篩選後：%d 欄（刪除 %d 欄）\n",
            ncol(X), sum(missing_rate >= 0.5)))

缺值率篩選後：318 欄（刪除 81 欄）


In [10]:
# ── 4. NZV (near zero variance predictors) 篩選 ──────────────────────────────────────────────
nzv_idx <- nearZeroVar(X)
if (length(nzv_idx) > 0) X <- X[, -nzv_idx]
cat(sprintf("NZV 篩選後：%d 欄（刪除 %d 欄）\n",
            ncol(X), length(nzv_idx)))

NZV 篩選後：282 欄（刪除 36 欄）


到這裡的時候，要分成3份。
- matrix --> pklm
- data.frame --> rb
- csv (加回 attrition) --> lgbm (需重新命名檔名)

In [187]:
# ── 5. 轉成 matrxi data.frame ────────────────────────
#' PKLM 需要轉 matrix，NA 保留 
X_mat <- as.matrix(X)
cat(sprintf("\n最終矩陣：%d × %d\n", nrow(X_mat), ncol(X_mat)))
cat(sprintf("剩餘 NA 數：%d\n", sum(is.na(X_mat))))

#' RB 需要轉 data frame
X_df <- as.data.frame(X)
cat(sprintf("\n最終資料框：%d × %d\n", nrow(X_df), ncol(X_df)))

#' 這裡務必記得改檔名
#data.frame(
#  #released_id = dat$release_id,
#  X_df,
#  attrition = dat$attrition
#) %>% write_csv('miss_lgbm.csv')


最終矩陣：1943 × 281
剩餘 NA 數：7744

最終資料框：1943 × 281


## Step 1 PLKM

這個比較久，先用 `pilot_pklmtest()` 粗估一下時間。

In [78]:
pilot_pklmtest(data=X_mat)

試跑中（num.proj=10, nrep=50）...
[1] "while done"
試跑耗時 2.1 秒
正式執行預估耗時 7–14 分鐘


In [156]:
# ── Step 2：正式執行 ─────────────────────────────────────────
cat("\n開始正式執行...\n")
start <- proc.time()

pvals_all <- PKLMtest(
  X_mat,
  num.proj              = 300,
  nrep                  = 500, #最終要500
  compute.partial.pvals = TRUE
)

elapsed <- (proc.time() - start)["elapsed"]
cat(sprintf("完成！實際耗時 %.1f 分鐘\n", elapsed / 60))



開始正式執行...
[1] "while done"
完成！實際耗時 17.5 分鐘


In [158]:
# ── Step 3：整理結果 ─────────────────────────────────────────
global_pval   <- pvals_all[1]
partial_pvals <- pvals_all[-1]
names(partial_pvals) <- colnames(X_mat)

cat(sprintf("\nGlobal p-value: %.4f  %s\n",
            global_pval,
            ifelse(global_pval < 0.05, "→ 拒絕 MCAR", "→ 無法拒絕 MCAR")))

# 輸出違反 MCAR 的變項
violators <- sort(partial_pvals[partial_pvals < 0.05])
cat(sprintf("違反 MCAR 的變項數：%d / %d\n",
            length(violators), length(partial_pvals)))



Global p-value: 0.0020  → 拒絕 MCAR
違反 MCAR 的變項數：281 / 281


In [160]:
tibble(
  name = names(violators),
  violators
) #|>
#write_csv('pklm72.csv')

# A tibble: 281 × 2
   name         violators
   <chr>            <dbl>
 1 baby_sex       0.00200
 2 relationship   0.00200
 3 cogc02         0.00200
 4 cogb01         0.00200
 5 cogc07         0.00200
 6 cogb02         0.00200
 7 cogb04         0.00200
 8 cogb05         0.00200
 9 cogb06         0.00200
10 cogb07         0.00200
# ℹ 271 more rows
# ℹ Use `print(n = ...)` to see more rows

## Step 2. RBtest

In [188]:
# ── 6. RBtest 執行 ───────────────────────────────────────────
cat("\n開始 RBtest（逐變項版）...\n")
t0 <- proc.time()

result <- RBtest(X_df)

elapsed <- (proc.time() - t0)["elapsed"]
cat(sprintf("完成！耗時 %.1f 秒\n", elapsed))



開始 RBtest（逐變項版）...


There were 18 warnings (use warnings() to see them)


完成！耗時 3.7 秒


In [189]:
# ── 7. 整理結果 ──────────────────────────────────────────────
# result 是一個向量：0 = MCAR, 1 = MAR, -1 = 完整（無缺失）
result_df <- data.frame(
  variable   = names(result$type),
  abs_nbrMD  = result$abs.nbrMD,
  rel_nbrMD  = result$rel.nbrMD,
  mechanism  = as.integer(result$type)
) %>%
  mutate(label = case_when(
    mechanism == -1 ~ "complete",
    mechanism ==  0 ~ "MCAR",
    mechanism ==  1 ~ "MAR"
  ))

n_complete <- sum(result_df$mechanism == -1)
n_mcar     <- sum(result_df$mechanism ==  0)
n_mar      <- sum(result_df$mechanism ==  1)

cat(sprintf("完整變項（無缺失）：%d\n", n_complete))
cat(sprintf("MCAR 變項：         %d\n", n_mcar))
cat(sprintf("MAR 變項：          %d\n", n_mar))


完整變項（無缺失）：71
MCAR 變項：         169
MAR 變項：          41


In [190]:
# MAR 變項清單（含缺失數量資訊）
mar_vars <- result_df %>%
  filter(label == "MAR") %>%
  arrange(desc(abs_nbrMD)) %>%
  select(variable, abs_nbrMD, rel_nbrMD)

if (nrow(mar_vars) > 0) {
  cat("\nMAR 變項（依缺失數排序，前 20）：\n")
  print(head(mar_vars, 20))
}

write.csv(result_df, "rbtest72.csv", row.names = FALSE)
cat("\n已存至 rbtest_results.csv\n")


MAR 變項（依缺失數排序，前 20）：
                 variable abs_nbrMD rel_nbrMD
fammar05         fammar05       376      0.19
fammar04         fammar04       375      0.19
famevic10       famevic10       251      0.13
pfb05b             pfb05b       218      0.11
pfb0501b         pfb0501b       215      0.11
pfb05a             pfb05a       209      0.11
pfb04               pfb04       208      0.11
fammar03         fammar03       180      0.09
fammar02         fammar02       179      0.09
care1               care1       142      0.07
fatherinvo05 fatherinvo05       133      0.07
fatherinvo01 fatherinvo01       131      0.07
fatherinvo04 fatherinvo04       131      0.07
fatherinvo02 fatherinvo02       130      0.07
fatherinvo03 fatherinvo03       130      0.07
pfa14               pfa14       122      0.06
pfb07               pfb07        86      0.04
pfa1501           pfa1501        84      0.04
motherinvo01 motherinvo01        77      0.04
motherinvo02 motherinvo02        77      0.04

已存至 rbtest_